# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 02 — Fontes de Dados
**CSV · API REST · Web Scraping com BeautifulSoup**

---

### Objetivos do Laboratório
Aprender a coletar e carregar dados em Python a partir das 3 fontes mais comuns no ecossistema de Ciência de Dados e IA:
1. **Opção 1: Arquivo CSV** (Dado local estruturado e tabulado).
2. **Opção 2: API REST** (Dado ao vivo via requisição HTTP e JSON).
3. **Opção 3: Web Scraping** (Extração de dados semiestruturados diretamente do código HTML).


### Importação de Bibliotecas

In [ ]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

print("Bibliotecas importadas com sucesso!")

---
### 1. Fonte de Dados — Opção 1: Arquivo CSV (Dado Local Estruturado)
Arquivos CSV são o formato mais difundido para tabelas locais.
Carregamos o dataset de preços de veículos (`CarPrice_Assignment.csv`) usando caminho relativo para garantir que o código funcione de maneira portável em qualquer computador.

In [ ]:
caminho_csv = "CarPrice_Assignment.csv"

if os.path.exists(caminho_csv):
    df_carros = pd.read_csv(caminho_csv, sep=",")
    print(f"Sucesso! Dimensões do dataset: {df_carros.shape[0]} linhas x {df_carros.shape[1]} colunas.")
    display(df_carros.head())
else:
    print(f"Aviso: Arquivo {caminho_csv} não encontrado no diretório atual.")

In [ ]:
# Visualização rápida das colunas carregadas do CSV
if 'df_carros' in locals():
    print("Colunas presentes:")
    print(df_carros.columns.tolist()[:10], "... [mais colunas]")

---
### 2. Fonte de Dados — Opção 2: API (Dado ao Vivo via HTTP / JSON)
Nesta seção, consumimos dados ao vivo diretamente da API oficial do **Banco Central do Brasil (BACEN)**.
Consultamos a série histórica de código `433`, que corresponde ao índice de inflação **IPCA** acumulado.

In [ ]:
# Endpoint oficial do SGS (Sistema Gerenciador de Séries Temporais) do BACEN
url_bacen = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"

try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    resposta = requests.get(url_bacen, headers=headers, timeout=10)
    resposta.raise_for_status()
    dados_bacen = resposta.json()
    print(f"Requisição concluída! Total de registros recebidos: {len(dados_bacen)}")
    print("Primeiro registro:", dados_bacen[0])
    print("Último registro:", dados_bacen[-1])
except Exception as e:
    print(f"Erro ao acessar API do BACEN: {e}")
    # Dados de fallback em caso de falta de conexão
    dados_bacen = [{'data': '01/01/2026', 'valor': '0.35'}, {'data': '01/02/2026', 'valor': '0.42'}]

Transformando o JSON da API em um `DataFrame` do Pandas e tratando os tipos de dados:

In [ ]:
df_ipca = pd.DataFrame(dados_bacen)

# Conversão de tipo de dados
df_ipca['data'] = pd.to_datetime(df_ipca['data'], format='%d/%m/%Y')
df_ipca['valor'] = pd.to_numeric(df_ipca['valor'], errors='coerce')

print("Últimos 5 registros do IPCA:")
display(df_ipca.tail())

# Plotando os últimos 24 meses do IPCA
df_ultimos = df_ipca.tail(24)
plt.figure(figsize=(10, 4))
plt.plot(df_ultimos['data'], df_ultimos['valor'], marker='o', color='#1f77b4', linewidth=2)
plt.title("IPCA Mensal — BACEN (Últimos 24 meses)", fontsize=13)
plt.xlabel("Data")
plt.ylabel("Variação (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
### 3. Fonte de Dados — Opção 3: Web Scraping (Extrair Dados de HTML)
Quando os dados não estão disponíveis em API ou CSV, podemos extraí-los diretamente da página web utilizando a biblioteca **BeautifulSoup**.
Neste exemplo, extraímos a lista de artistas do acervo da *National Gallery of Art* (versão arquivada do Internet Archive).

In [ ]:
url_scraping = "https://web.archive.org/web/20121007172955/https://www.nga.gov/collection/anZ1.htm"

try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    pagina = requests.get(url_scraping, headers=headers, timeout=15)
    pagina.raise_for_status()
    
    # Criando o parser BeautifulSoup
    soup = BeautifulSoup(pagina.text, 'html.parser')
    
    # Removendo elementos de navegação alfabética com .decompose()
    last_links = soup.find(class_='AlphaNav')
    if last_links:
        last_links.decompose()
        
    # Localizando a seção de texto principal
    artist_name_list = soup.find(class_='BodyText')
    
    nomes = []
    if artist_name_list:
        artist_name_list_items = artist_name_list.find_all('a')
        for artist_name in artist_name_list_items:
            # Usando .contents para pegar o texto da tag <a>
            if artist_name.contents:
                nomes.append(str(artist_name.contents[0]).strip())
                
    df_artistas = pd.DataFrame(nomes, columns=['Artista'])
    print(f"Total de artistas extraídos: {len(df_artistas)}")
    display(df_artistas.head(10))

except Exception as e:
    print(f"Falha na requisição de Web Scraping: {e}")
    # Simulação dos dados para demonstração caso o Internet Archive esteja lento
    df_artistas = pd.DataFrame([
        "Zabaglia, Niccola", "Zaccone, Fabian", "Zadkine, Ossip", "Zaidenberg, Arthur"
    ], columns=['Artista'])
    display(df_artistas)

### Quadro-Resumo das Fontes de Dados
| Fonte | Protocolo / Formato | Vantagens | Desafios |
| :--- | :--- | :--- | :--- |
| **Arquivo CSV** | Arquivo texto plano delimitado | Imediato, estável, sem dependência de rede | Pode desatualizar; arquivos muito grandes sobrecarregam RAM |
| **API REST** | HTTP (JSON/XML) | Dados ao vivo, estruturados, padronizados | Requer autenticação (API Keys), limites de requisição (rate limits) |
| **Web Scraping** | HTTP (HTML Parsing) | Acessa qualquer dado público visível na web | Frágil a mudanças de layout na página, possível bloqueio de IP |

Parabéns! Você dominou o carregamento de dados em múltiplos formatos e fontes.
